# **Experiment-5: Subword Level Tokenization and POS Tagging**

This experiment covers:
- **5.1**: BPE Tokenization (pretrained & from scratch)
- **5.2**: SentencePiece Tokenization (pretrained & from scratch)
- **5.3**: POS Tagging (spaCy & NLTK)
- **5.4**: POS Tagging with Token Frequency

In [ ]:
pip install --upgrade typing_extensions pydantic pydantic-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.1/488.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.3 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.5
    Uninstalling pydantic_core-2.46.5:
      Successfully uninstalled pydantic_core-2.46.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.5
    Uninstalling pydantic-2.13.5:
      Successfully uninstalled pydantic-2.13.5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

# BPE Tokenization (HuggingFace tokenizers library)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Pretrained tokenizers (HuggingFace transformers)
from transformers import AutoTokenizer

# SentencePiece tokenizer (Google)
import sentencepiece as spm

# POS Tagging
import spacy
import nltk
from collections import Counter

# Download required NLTK data files
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
# ============================================================
# LOAD INPUT DATA
# ============================================================

file_path = "/content/drive/MyDrive/NLP_Lab/Experiment-5/input_sub_word_data.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

# Show a preview of the corpus
print("Input Data (first 500 characters):")
print("=" * 50)
print(text[:500])
print("...")

Input Data (first 500 characters):
Natural language processing is a branch of artificial intelligence that
enables computers to understand, interpret, and generate human language.
Natural language processing combines linguistics, computer science, and
machine learning to process large amounts of natural language data.

Tokenization is one of the fundamental steps in natural language processing.
A tokenizer divides text into smaller units called tokens. Tokens may be
words, characters, or subword units. Word-level tokenization is 
...


In [ ]:
# ============================================================
# HELPER FUNCTION: Display tokens neatly
# ============================================================

def display_tokens(tokens, token_ids, title, max_display=20):
    """Print tokens and their IDs in a formatted table.

    Args:
        tokens: List of token strings
        token_ids: List of corresponding integer IDs
        title: Section heading to print
        max_display: Number of tokens to show (default 20)
    """
    print(f"\n{title}")
    print(f"{'Token':30} {'Token ID'}")
    print("-" * 45)

    for token, tid in list(zip(tokens, token_ids))[:max_display]:
        print(f"{token:30} {tid}")

    if len(tokens) > max_display:
        print(f"\n... ({len(tokens) - max_display} more tokens not shown)")

    print(f"\nTotal tokens: {len(tokens)}")
    print(f"Unique tokens: {len(set(tokens))}")

---
## Experiment 5.1: BPE Tokenization

Tokenize text into subword units using BPE. Display tokens and their IDs.

**Two approaches:** (a) Pretrained model, (b) Trained from scratch

In [ ]:
# ============================================================
# 5.1(a) BPE — Using Pretrained GPT-2 Tokenizer
# ============================================================
# GPT-2's tokenizer was trained on ~40GB of internet text.
# It has a vocabulary of ~50,257 subword tokens.
# The 'G' prefix (shown as Ġ) means there's a leading space.

bpe_tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokens = bpe_tokenizer.tokenize(text)
token_ids = bpe_tokenizer.convert_tokens_to_ids(tokens)

display_tokens(tokens, token_ids, "BPE Tokenization — Pretrained GPT-2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1348 > 1024). Running this sequence through the model will result in indexing errors



BPE Tokenization — Pretrained GPT-2
Token                          Token ID
---------------------------------------------
Natural                        35364
Ġlanguage                      3303
Ġprocessing                    7587
Ġis                            318
Ġa                             257
Ġbranch                        8478
Ġof                            286
Ġartificial                    11666
Ġintelligence                  4430
Ġthat                          326
Ċ                              198
en                             268
ables                          2977
Ġcomputers                     9061
Ġto                            284
Ġunderstand                    1833
,                              11
Ġinterpret                     6179
,                              11
Ġand                           290

... (1328 more tokens not shown)

Total tokens: 1348
Unique tokens: 481


In [ ]:
# ============================================================
# 5.1(b) BPE — Trained from Scratch on Our Corpus
# ============================================================
# We train a BPE tokenizer with vocab_size=500 on our small corpus.
# min_frequency=2 means only character pairs appearing >=2 times get merged.

bpe = Tokenizer(BPE(unk_token="[UNK]"))
bpe.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
)

bpe.train(files=[file_path], trainer=trainer)

output = bpe.encode(text)

display_tokens(output.tokens, output.ids, "BPE Tokenization — Trained from Scratch")
print(f"Vocabulary Size: {bpe.get_vocab_size()}")


BPE Tokenization — Trained from Scratch
Token                          Token ID
---------------------------------------------
Natural                        359
language                       142
processing                     192
is                             82
a                              24
b                              25
r                              41
an                             54
ch                             84
of                             89
ar                             59
t                              43
ific                           444
ial                            380
in                             51
t                              43
el                             105
l                              35
ig                             307
ence                           178

... (1588 more tokens not shown)

Total tokens: 1608
Unique tokens: 394
Vocabulary Size: 500


---
## Experiment 5.2: SentencePiece Tokenization

Tokenize text into subword units using SentencePiece. Display tokens and their IDs.

**Two approaches:** (a) Pretrained model, (b) Trained from scratch

In [ ]:
# ============================================================
# 5.2(a) SentencePiece — Using Pretrained mT5-small
# ============================================================
# mT5 (multilingual T5) uses SentencePiece with 250,000 tokens.
# The '▁' prefix represents a leading space.

sp_tokenizer = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

sp_tokens = sp_tokenizer.tokenize(text)
sp_token_ids = sp_tokenizer.convert_tokens_to_ids(sp_tokens)

display_tokens(sp_tokens, sp_token_ids, "SentencePiece Tokenization — Pretrained mT5")

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]


SentencePiece Tokenization — Pretrained mT5
Token                          Token ID
---------------------------------------------
▁Natural                       10630
▁language                      17896
▁processing                    36031
▁is                            339
▁                              259
a                              262
▁branch                        69608
▁of                            304
▁artificial                    57871
▁                              259
intelligence                   73171
▁that                          533
▁en                            289
ables                          8924
▁computers                     108310
▁to                            288
▁understand                    16571
,                              261
▁interpret                     25224
,                              261

... (1537 more tokens not shown)

Total tokens: 1557
Unique tokens: 405


In [ ]:
# ============================================================
# 5.2(b) SentencePiece — Trained from Scratch
# ============================================================
# We train our own SentencePiece model with vocab_size=300.
# character_coverage=1.0 means include all characters from the corpus.

spm.SentencePieceTrainer.train(
    input=file_path,
    model_prefix="my_sentencepiece",
    vocab_size=300,
    model_type="bpe",
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor(model_file="my_sentencepiece.model")

sp_tokens = sp.encode(text, out_type=str)
sp_token_ids = sp.encode(text, out_type=int)

display_tokens(sp_tokens, sp_token_ids, "SentencePiece Tokenization — Trained from Scratch")
print(f"Vocabulary Size: {sp.get_piece_size()}")


SentencePiece Tokenization — Trained from Scratch
Token                          Token ID
---------------------------------------------
▁N                             213
atural                         174
▁language                      118
▁processing                    178
▁is                            84
▁a                             7
▁b                             32
r                              259
an                             15
ch                             210
▁of                            58
▁a                             7
r                              259
t                              258
if                             75
ic                             46
i                              262
al                             21
▁int                           156
el                             61

... (2311 more tokens not shown)

Total tokens: 2331
Unique tokens: 266
Vocabulary Size: 300


---
## Experiment 5.3: POS Tagging

Perform Part-of-Speech tagging on:

> *"The young student is reading an interesting book in the library."*

**Two approaches:** (a) spaCy, (b) NLTK

In [ ]:
# Define the sentence for POS tagging
pos_text = "The young student is reading an interesting book in the library."

In [ ]:
# ============================================================
# 5.3(a) POS Tagging — Using spaCy
# ============================================================
# spaCy uses a neural network model for POS tagging.
# .pos_ gives the Universal POS tag (NOUN, VERB, ADJ, etc.)
# spacy.explain() converts the tag to a readable description.

nlp = spacy.load("en_core_web_sm")
doc = nlp(pos_text)

print("POS Tagging using spaCy\n")
print(f"{'Token':20} {'POS Tag':10} {'Description'}")
print("-" * 60)

for token in doc:
    print(f"{token.text:20} {token.pos_:10} {spacy.explain(token.pos_)}")

POS Tagging using spaCy

Token                POS Tag    Description
------------------------------------------------------------
The                  DET        determiner
young                ADJ        adjective
student              NOUN       noun
is                   AUX        auxiliary
reading              VERB       verb
an                   DET        determiner
interesting          ADJ        adjective
book                 NOUN       noun
in                   ADP        adposition
the                  DET        determiner
library              NOUN       noun
.                    PUNCT      punctuation


In [ ]:
# ============================================================
# 5.3(b) POS Tagging — Using NLTK
# ============================================================
# NLTK uses the Averaged Perceptron tagger.
# It gives Penn Treebank tags (DT, JJ, NN, VBZ, VBG, IN, etc.)

nltk_tokens = nltk.word_tokenize(pos_text)
nltk_tags = nltk.pos_tag(nltk_tokens)

print("POS Tagging using NLTK\n")
print(f"{'Token':20} {'POS Tag'}")
print("-" * 40)

for token, tag in nltk_tags:
    print(f"{token:20} {tag}")

POS Tagging using NLTK

Token                POS Tag
----------------------------------------
The                  DT
young                JJ
student              NN
is                   VBZ
reading              VBG
an                   DT
interesting          JJ
book                 NN
in                   IN
the                  DT
library              NN
.                    .


---
## Experiment 5.4: POS Tagging with Token Frequency

Tag the full corpus with POS tags and display each unique token's tag, description, and frequency.

In [ ]:
# ============================================================
# 5.4 POS Tagging with Frequency — Using spaCy
# ============================================================

doc = nlp(text)

# Count how often each lowercased token appears (skip whitespace)
frequency = Counter(
    token.text.lower()
    for token in doc
    if not token.is_space
)

# Collect unique token info (POS, description, frequency)
token_info = {}
for token in doc:
    if token.is_space:
        continue
    word = token.text.lower()
    if word not in token_info:
        token_info[word] = {
            "POS": token.pos_,
            "Description": spacy.explain(token.pos_),
            "Frequency": frequency[word]
        }

# Display results
print("POS Tagging with Frequency — spaCy\n")
print(f"{'Token':20} {'POS Tag':10} {'Description':20} {'Frequency'}")
print("-" * 75)

for word, info in token_info.items():
    print(
        f"{word:20} "
        f"{info['POS']:10} "
        f"{str(info['Description']):20} "
        f"{info['Frequency']}"
    )

print(f"\nTotal unique tokens: {len(token_info)}")

POS Tagging with Frequency — spaCy

Token                POS Tag    Description          Frequency
---------------------------------------------------------------------------
natural              ADJ        adjective            6
language             NOUN       noun                 12
processing           NOUN       noun                 7
is                   AUX        auxiliary            14
a                    DET        determiner           28
branch               NOUN       noun                 1
of                   ADP        adposition           22
artificial           ADJ        adjective            1
intelligence         NOUN       noun                 1
that                 PRON       pronoun              4
enables              VERB       verb                 1
computers            NOUN       noun                 1
to                   PART       particle             15
understand           VERB       verb                 4
,                    PUNCT      punctuation       

---
## Experiment 5.5: Word2Vec CBOW Model

Train a Word2Vec Continuous Bag of Words (CBOW) model using a small corpus, display the vector representation of a selected word, and find the three most similar words.

In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 56.5 MB/s eta 0:00:00


In [ ]:
# Write corpus.txt
corpus_content = """Natural language processing is interesting
Natural language processing is useful
I love machine learning
Machine learning is useful
Deep learning is a part of machine learning
Artificial intelligence is changing the world
Natural language processing uses machine learning
Deep learning and artificial intelligence are related
I study natural language processing
Machine learning helps artificial intelligence"""

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(corpus_content)

print("corpus.txt created successfully.")

corpus.txt created successfully.


In [ ]:
import nltk
from gensim.models import Word2Vec

# Read and tokenize the corpus
with open("corpus.txt", "r", encoding="utf-8") as f:
    corpus_sentences = [line.strip().lower() for line in f if line.strip()]

tokenized_sentences = [nltk.word_tokenize(sentence) for sentence in corpus_sentences]

# Train CBOW Word2Vec model (sg=0 specifies CBOW)
cbow_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=0,
    seed=42
)

selected_word = "learning"
print(f"Vector representation of '{selected_word}':\n")
print(cbow_model.wv[selected_word])

similar_words = cbow_model.wv.most_similar(selected_word, topn=3)
print(f"\n3 most similar words to '{selected_word}':")
for word, score in similar_words:
    print(f"{word}: {score:.4f}")

Vector representation of 'learning':

[-1.64194256e-02  1.09623624e-02  6.18664082e-03 -2.44291453e-03
 -2.67175213e-03  1.43442200e-02 -1.65647212e-02  7.90307485e-03
 -1.19350813e-02 -1.62306819e-02  1.06711825e-03  1.90218762e-02
  9.42538120e-03  1.04379542e-02  8.69894400e-03  1.14436746e-02
  5.25794807e-04 -1.48803545e-02  1.35905892e-02 -1.98469963e-03
  1.87702262e-05 -5.17611997e-03 -1.26957661e-02  1.70679856e-02
  1.12640820e-02  5.74814342e-03 -3.89753468e-03  1.29009550e-02
  1.82045123e-03 -2.26977956e-03 -1.98267330e-03 -1.09046511e-02
 -1.63235124e-02  2.18333839e-03  1.55115416e-02 -1.74367893e-02
  1.43361948e-02  1.30947987e-02 -8.92965496e-03  5.27681690e-03
 -1.33802714e-02  1.03321355e-02  8.02171696e-03 -5.81205077e-03
 -1.72761176e-02  1.88240912e-02 -2.16680439e-03  1.57325435e-02
  7.11806817e-03  1.11352745e-02]

3 most similar words to 'learning':
uses: 0.2422
useful: 0.1310
changing: 0.1283


---
## Experiment 5.6: Word2Vec Skip-Gram Model

Train a Word2Vec Skip-Gram model using the same corpus, retrieve the vector for 'language', and find the five most similar words.

In [ ]:
# Train Skip-Gram Word2Vec model (sg=1 specifies Skip-Gram)
sg_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    seed=42
)

target_word = "language"
print(f"Vector representation of '{target_word}':\n")
print(sg_model.wv[target_word])

sg_similar_words = sg_model.wv.most_similar(target_word, topn=5)
print(f"\n5 most similar words to '{target_word}':")
for word, score in sg_similar_words:
    print(f"{word}: {score:.4f}")

Vector representation of 'language':

[-0.00533336  0.01633794 -0.00019166  0.00799399 -0.00169816 -0.00935986
  0.01056546  0.01876536 -0.0094546   0.01115877 -0.0095355   0.00868602
  0.01154729 -0.00201818  0.00947814 -0.00910828 -0.0168252  -0.01614576
 -0.00212467  0.01609557 -0.0148905  -0.00178489  0.00816213 -0.01191122
  0.00887454 -0.00776599  0.01234394  0.00315739  0.00187843 -0.01292972
 -0.00146346  0.01427294 -0.0192307   0.01034501 -0.00028703  0.00879778
  0.00652633 -0.00273675 -0.00765842  0.00510086 -0.0143827   0.00337547
 -0.01619635  0.00599824  0.00516615 -0.01663174  0.0103059  -0.00336611
  0.01147311 -0.01834095]

5 most similar words to 'language':
uses: 0.2865
love: 0.2048
helps: 0.1944
intelligence: 0.1274
of: 0.1155


---
## Experiment 5.7: Cosine Similarity between Word Pairs

Calculate the cosine similarity between selected word pairs using the trained Word2Vec models.

In [ ]:
word_pairs = [
    ("machine", "learning"),
    ("natural", "language"),
    ("deep", "learning")
]

print("Cosine Similarity (CBOW Model):")
print("-" * 40)
for w1, w2 in word_pairs:
    similarity = cbow_model.wv.similarity(w1, w2)
    print(f"{w1} <-> {w2} : {similarity:.4f}")

print("\nCosine Similarity (Skip-Gram Model):")
print("-" * 40)
for w1, w2 in word_pairs:
    similarity = sg_model.wv.similarity(w1, w2)
    print(f"{w1} <-> {w2} : {similarity:.4f}")

Cosine Similarity (CBOW Model):
----------------------------------------
machine <-> learning : -0.2329
natural <-> language : -0.0297
deep <-> learning : -0.0983

Cosine Similarity (Skip-Gram Model):
----------------------------------------
machine <-> learning : -0.2330
natural <-> language : -0.0296
deep <-> learning : -0.0978
